In [3]:
import os, sys
REPO = "https://github.com/salsyaniditiani-commits/simuteh--2330511061-.git"

if not os.path.exists("/content/simuteh"):
    !git clone {REPO} /content/simuteh
os.chdir("/content/simuteh")

for f in ["notebook", "hasil", "laporan"]:
    os.makedirs(f, exist_ok=True)

if not os.path.exists("data/sintetis/citra"):
    !python src/buat_dataset_sintetis.py

os.chdir("/content/simuteh/notebook")
sys.path.append("../src")

print("Folder kerja:", os.getcwd())
print("Jumlah citra:", len(os.listdir("../data/sintetis/citra")))
print("Isi src:", os.listdir("../src"))

Folder kerja: /content/simuteh/notebook
Jumlah citra: 60
Isi src: ['__pycache__', 'praproses.py', 'utils_cv.py', 'buat_dataset_sintetis.py']


kode 5.1

In [4]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from utils_cv import baca, statistik, tampil, kontras_bercak
from praproses import praproses, simpan_param

# Parameter praproses
param = {
    "gamma": 1.0,
    "clahe_clip": 2.0,
    "penapis": "median",
    "kernel": 3
}

simpan_param(param, "../hasil/param_praproses.json")

# Memilih kanal terbaik berdasarkan Bagian 2
KANAL = 2  # Indeks BGR: 0 = B, 1 = G, 2 = R

# Membaca dataset
label = pd.read_csv("../data/sintetis/label.csv")
baris = []

for _, d in label[label.kelas != "sehat"].iterrows():
    img = baca(f"../data/sintetis/citra/{d.nama_file}")
    md = cv2.imread(f"../data/sintetis/mask_daun/{d.nama_file}", 0)
    mb = cv2.imread(f"../data/sintetis/mask_bercak/{d.nama_file}", 0)

    # Melakukan praproses
    out = praproses(img, param)

    # Menghitung kontras bercak sebelum dan sesudah praproses
    baris.append([
        d.kondisi_cahaya,
        kontras_bercak(img[:, :, KANAL], md, mb),
        kontras_bercak(out[:, :, KANAL], md, mb)
    ])

# Membuat tabel evaluasi
evaluasi = pd.DataFrame(
    baris,
    columns=["kondisi", "KB_sebelum", "KB_sesudah"]
)

print(evaluasi.groupby("kondisi").mean().round(2))

                KB_sebelum  KB_sesudah
kondisi                               
bayangan              4.83        8.03
gelap                 8.49       14.42
kontras_rendah        3.54        4.65
normal                9.05       10.05
terang                6.27        6.01
